# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring** — a continuous, per-page "how far below its position-tier norm is this page sitting" number, consumed downstream as a **ranking** (the review queue is just those scores sorted, highest opportunity first).

Why not the other three task types:

- **Not classification.** There's no clean binary "needs review / doesn't" label in the data — only a continuous CTR gap. Forcing that into a yes/no bucket means inventing a threshold, and that threshold *is* the interesting part of the problem, not a preprocessing step to throw away. (The lane guide does allow classification "if you define a leakage-safe label" — I'm not ruling it out permanently, just not starting there, since I don't yet have a trustworthy binary ground truth.)
- **Not clustering.** Clustering would group pages into unlabeled segments ("archetypes") — useful for a different question (that's literally Lane 3), but it doesn't hand a reviewer a single number to sort by, and it doesn't answer "which page first."
- **Scoring, surfaced as ranking**, matches the actual decision from Week 1: a reviewer with limited time works down a sorted list. The model's job is to estimate one number per page well enough that the sort order is trustworthy — that's a scoring/regression problem under the hood, not a pairwise ranking-loss problem (I don't have query-level competing-pages-for-the-same-slot structure here, just independent pages).

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Same "visible page" filter as Week 1 -- real search position + enough impressions to trust CTR.
lane_df = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 500)].copy()
print(f"Lane slice: {len(lane_df):,} visible pages out of {len(df):,} total rows")

Lane slice: 16,726 visible pages out of 30,000 total rows


## 2. Target or proxy

**What I'd predict:** `ctr_gap` — the difference between a page's actual CTR and the CTR its position tier normally earns:

```
ctr_gap = ctr − expected_ctr(position_tier)
```

The simplest version of `expected_ctr` is the tier's median CTR (what I used in Week 1 to size the opportunity). A stronger version — the thing worth testing a model against — would condition the expectation on more than just tier: content_type, main_intent, word_count_tier, freshness_tier, etc., since Section 5 below shows those shift the norm too.

**Where this label comes from — and why I have to be honest about it:** `ctr_gap` is a **defined-rule proxy**, not an observed outcome. Nobody in this dataset has a column that says "this page was under-capturing and a fix would have worked." I don't have an experiment table (no A/B test flag, no before/after intervention record) — so the target I can build is a *statistical* gap (how unusual is this page's CTR for its context), not a *causal* one (would fixing the title actually move it). Those are different things, and Section 5 of Week 1's notebook already commits to never blurring that line. This proxy is a reasonable stand-in because a persistent, large, well-powered gap is at least a necessary condition for an opportunity to exist — but it isn't sufficient proof one does.

In [5]:
# Target/proxy: ctr_gap = observed CTR - this page's position-tier median CTR.
# Simplest possible "expected_ctr" -- the floor any model has to beat.
tier_median_ctr = lane_df.groupby('position_tier')['ctr'].transform('median')
lane_df['expected_ctr_tier_median'] = tier_median_ctr
lane_df['ctr_gap'] = lane_df['ctr'] - lane_df['expected_ctr_tier_median']

print("What the proxy target looks like, row by row:")
lane_df[['position_tier', 'ctr', 'expected_ctr_tier_median', 'ctr_gap']].head(8)

What the proxy target looks like, row by row:


,position_tier,ctr,expected_ctr_tier_median,ctr_gap
0,striking,0.76,0.17,0.59
1,page_3_5,0.05,0.09,-0.04
2,page_3_5,0.09,0.09,0.00
3,page_1,0.49,0.24,0.25
4,page_3_5,0.13,0.09,0.04
5,page_1,0.03,0.24,-0.21
7,page_3_5,0.06,0.09,-0.03
8,page_3_5,0.09,0.09,0.00


## 3. Success metric

**The metric I'd actually want, eventually:** Precision@K — of the top K pages the model surfaces in a given week (K sized to a reviewer's real weekly capacity, e.g. 20–50), what fraction turn out, on human review, to be genuine, fixable opportunities? This is the metric that matches the real decision: reviewer time is scarce, so the top of the list has to be right far more than the bottom does. I don't have human-reviewed labels yet — that requires an actual reviewer sitting with the queue, which is downstream work — so I can't compute Precision@K from the starter data today.

**What I *can* honestly compute right now, as an interim sanity check:** the starter dataset already carries two trailing windows (`*_prev_30d` and `*_last_30d`), which lets me build a leakage-safe forward check: compute the gap score using only the *earlier* window, then see whether it's associated with how CTR actually moved in the *later* window. If the gap score is measuring nothing real, there should be no relationship at all.

This is **not** a substitute for Precision@K, and it is **not** causal — it's a directional plausibility check, computed below, before I've built any model at all.

In [6]:
# Interim, honest, non-causal check: does a gap computed on the EARLIER window (prev_30d)
# say anything about how CTR actually moved in the LATER window (last_30d)?
# This is leakage-safe -- the gap never sees the outcome window it's being checked against.
checkable = lane_df[
    (lane_df['impressions_prev_30d'] >= 100) & (lane_df['impressions_last_30d'] >= 100)
].copy()

checkable['ctr_prev_30d'] = checkable['clicks_prev_30d'] / checkable['impressions_prev_30d']
checkable['ctr_last_30d'] = checkable['clicks_last_30d'] / checkable['impressions_last_30d']

tier_median_prev = checkable.groupby('position_tier')['ctr_prev_30d'].transform('median')
checkable['gap_prev_30d'] = checkable['ctr_prev_30d'] - tier_median_prev
checkable['ctr_delta'] = checkable['ctr_last_30d'] - checkable['ctr_prev_30d']

rho, pval = spearmanr(checkable['gap_prev_30d'], checkable['ctr_delta'])
print(f"Pages with enough volume in both windows: {len(checkable):,}")
print(f"Spearman correlation, prior-window gap vs. later CTR movement: rho={rho:.3f} (p={pval:.2e})")

# Same check, framed the way a reviewer would actually read it: does the bottom slice
# (biggest apparent under-capture) move differently from the top slice (already above norm)?
q20, q80 = checkable['gap_prev_30d'].quantile([0.20, 0.80])
bottom = checkable[checkable['gap_prev_30d'] <= q20]
top = checkable[checkable['gap_prev_30d'] >= q80]
print()
print(f"Bottom quintile (n={len(bottom):,}, biggest under-capture): "
      f"{(bottom['ctr_delta'] > 0).mean()*100:.1f}% saw CTR rise afterward")
print(f"Top quintile    (n={len(top):,}, already above tier norm): "
      f"{(top['ctr_delta'] > 0).mean()*100:.1f}% saw CTR rise afterward")
print()
print("Reading this carefully: the negative correlation is expected and partly mechanical")
print("(regression to the mean -- pages far from their tier's median tend to drift back toward it")
print("either direction). It is NOT proof that a title/meta rewrite would work. What it does show")
print("is that the gap score isn't random noise -- it's tracking something that actually moves,")
print("which is the honest bar an interim metric can clear before Precision@K becomes available.")

Pages with enough volume in both windows: 15,088
Spearman correlation, prior-window gap vs. later CTR movement: rho=-0.284 (p=2.78e-278)

Bottom quintile (n=3,154, biggest under-capture): 48.7% saw CTR rise afterward
Top quintile    (n=3,018, already above tier norm): 37.4% saw CTR rise afterward

Reading this carefully: the negative correlation is expected and partly mechanical
(regression to the mean -- pages far from their tier's median tend to drift back toward it
either direction). It is NOT proof that a title/meta rewrite would work. What it does show
is that the gap score isn't random noise -- it's tracking something that actually moves,
which is the honest bar an interim metric can clear before Precision@K becomes available.


## 4. The unit of analysis, as a real dataframe

One row = **one (client_id, content_id) pair** — a single published content item, summarized over a trailing window. Same grain as Week 1, filtered the same way: a "visible" page needs a real recorded search position and at least 500 impressions in the last 90 days, or the CTR underneath it is too noisy to trust.

In [7]:
# The unit of analysis, as an actual dataframe: one row = one (client_id, content_id) pair.
cols = ['client_id', 'content_id', 'position_tier', 'avg_position',
        'impressions_90d', 'clicks_90d', 'ctr', 'content_type', 'main_intent', 'ctr_gap']
lane_df[cols].head(10)

,client_id,content_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,content_type,main_intent,ctr_gap
0,client_f369cb89fc,content_304f48230142,striking,10.6,3803,29,0.76,keyword article,transactional,0.59
1,client_4e07408562,content_a1fb4e703a9e,page_3_5,20.3,15320,7,0.05,keyword article,informational,-0.04
2,client_7f2253d7e2,content_9aa793d4d895,page_3_5,36.5,12581,11,0.09,keyword article,informational,0.00
3,client_19581e27de,content_331d6c4de07b,page_1,6.2,11751,58,0.49,keyword article,commercial,0.25
4,client_3fdba35f04,content_d99b7a2d90ca,page_3_5,44.0,19140,24,0.13,keyword article,informational,0.04
5,client_f369cb89fc,content_d4084a4bc775,page_1,8.5,3970,1,0.03,keyword article,transactional,-0.21
7,client_19581e27de,content_a63219c6e95a,page_3_5,21.2,1724,1,0.06,keyword article,commercial,-0.03
8,client_6208ef0f77,content_5e6c160719bc,page_3_5,46.0,32574,29,0.09,keyword article,informational,0.00
9,client_19581e27de,content_c27558df2b0c,page_1,4.9,1240,2,0.16,keyword article,informational,-0.08
10,client_19581e27de,content_d8ee6cc6d642,top_3,2.2,20919,324,1.55,keyword article,commercial,1.35


## 5. Why ML beats a fixed rule here

Week 1 already showed position tier alone swings median CTR from 0.00 to 0.24 — that's the first axis a rule has to respect. The code below shows a **second** axis hiding inside the *same* tier: `content_type` and `main_intent` shift the norm again, on top of position. Within page_1 alone, median CTR runs from about 0.24 for keyword articles up near 0.26 for transactional-intent pages and down toward 0.22 for commercial-intent pages — real, if modest, movement layered on top of the tier effect.

That's the case against a fixed rule: to hand-encode "expected CTR" honestly, I'd need a branch for every `position_tier × content_type × main_intent` combination (5 tiers × 3 content types × 4 intents = up to 60 cells), many of them thin on data (comparison articles at page_1 are only 50 rows), and I still wouldn't have touched freshness, word count, or age — each of which plausibly nudges the norm too. Nested if-statements don't scale past two or three interacting variables before they become unreadable and overfit to whichever combination happened to show up in this snapshot.

A learned model (starting simple — linear regression with those variables as features, moving to gradient-boosted trees only if it clearly earns its complexity) can absorb all of these interacting factors into one smooth, calibrated estimate of "expected CTR given everything I know about this page," and can be re-fit as new reason codes or features get added. The rule isn't wrong to exist — the tier-median baseline is still the thing any model has to beat — but it's the floor, not the ceiling.

In [8]:
# A second axis of variation hiding inside a single position tier: content_type and main_intent.
page1 = lane_df[lane_df['position_tier'] == 'page_1']

print("Within page_1 alone, median CTR by content_type:")
print(page1.groupby('content_type')['ctr'].agg(n='size', median_ctr='median'))

print()
print("Within page_1 alone, median CTR by main_intent:")
print(page1.groupby('main_intent')['ctr'].agg(n='size', median_ctr='median'))

n_tiers = lane_df['position_tier'].nunique()
n_types = lane_df['content_type'].nunique()
n_intents = lane_df['main_intent'].nunique()
print(f"\nA fixed rule covering just these three fields needs up to "
      f"{n_tiers} x {n_types} x {n_intents} = {n_tiers*n_types*n_intents} branches -- "
      f"before word count, freshness, or age even enter the picture.")

Within page_1 alone, median CTR by content_type:
                       n  median_ctr
content_type                        
comparison article    50        0.00
feedly article        83        0.26
keyword article     6931        0.24

Within page_1 alone, median CTR by main_intent:
                  n  median_ctr
main_intent                    
commercial     1210       0.220
informational  4012       0.230
navigational      8       0.325
transactional  1734       0.260

A fixed rule covering just these three fields needs up to 5 x 3 x 4 = 60 branches -- before word count, freshness, or age even enter the picture.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.